# 📓 Semana 2 · Dia 4 — Modelagem Dimensional Completa — Curso Intensivo

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (modelagem + SQL), DEP (SCD/CDC) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Modelo dimensional completo + Bus Matrix do projeto |

---


## 📖 Teoria — 1. Por que modelar? O pecado original dos dados não modelados

Imagine um armazém onde cada caixa chega com etiqueta diferente, sem corredor, sem prateleira. Você até guarda tudo — mas **ninguém acha nada**, e quando acha, o número não bate com o do colega. É exatamente o que acontece quando dados vão do Bronze direto para dashboards sem modelagem.

**Sintomas de um modelo ausente**:
- 10 dashboards, 10 números diferentes para 'receita do mês'.
- Qualquer pergunta nova exige um engenheiro escrever SQL complexo.
- Uma coluna muda de nome e 30 relatórios quebram.

A **modelagem dimensional** (Ralph Kimball, anos 90) resolve isso separando o **evento** (o fato que aconteceu) do **contexto** (quem, o que, onde, quando). Essa separação é a base de todo BI, de todo Data Warehouse moderno — e da camada **Prata/Ouro** da Medallion.


## 📖 Teoria — 2. OLTP vs OLAP — duas gravidades diferentes

| Aspecto | OLTP (transacional) | OLAP (analítico) |
|---|---|---|
| Objetivo | Registrar a operação | Entender o negócio |
| Operação típica | `INSERT 1 pedido` | `SUM(receita) por país no ano` |
| Modelo | Normalizado (3FN), sem redundância | Denormalizado (Star), redundância proposital |
| Leitura vs Escrita | 1 linha por vez, muita escrita | Milhões de linhas, quase só leitura |
| Exemplo | Postgres do e-commerce (pedidos) | Lakehouse `fato_vendas` (análise) |

**Regra de ouro**: NUNCA faça BI direto no OLTP. O OLAP é uma **cópia modelada** otimizada para perguntas. Na Medallion: OLTP → Bronze (cópia crua) → Prata/Ouro (modelo OLAP).


## 📖 Teoria — 3. Inmon vs Kimball — e onde a Medallion se posiciona

- **Inmon (top-down)**: modela tudo normalizado (3FN) primeiro, depois cria marts. Rigoroso, lento.
- **Kimball (bottom-up)**: começa pelos **processos de negócio**, entrega um Star por vez, integra por **dimensões conformadas**. Ágil, incremental.

**A Medallion é Kimball moderna**:
- Bronze = staging/raw (como a área de stage do Kimball).
- Prata = dimensões e fatos limpos, **conformed**, SCD — o Data Warehouse dimensional.
- Ouro = marts e agregados denormalizados — os Data Marts.

Ou seja: Kimball descreve **o que modelar**; Medallion descreve **onde guardar cada camada** no Lakehouse. Usamos os dois juntos.


## 📖 Teoria — 4. Os 4 passos do design dimensional (Kimball) — o método

Todo modelo dimensional nasce de 4 decisões, **nesta ordem**:

1. **Escolher o processo de negócio** — ex.: 'Vendas', 'Estoque', 'Atendimento'.
2. **Declarar o grain** — a granularidade de UMA linha da fato. Ex.: 'uma linha por item de nota'.
3. **Escolher as dimensões** — o contexto: quem comprou, o que, quando, onde, como pagou.
4. **Identificar os fatos** — as medidas numéricas: quantidade, preço, desconto, receita.

> ⚠️ Se errar o grain, todo o resto fica errado. Por isso o passo 2 merece uma seção inteira.


## 📖 Teoria — 5. Grain — a decisão #1 do projeto (e a mais cara se errar)

**Grain** = o que UMA linha da fato representa. É a 'frase' do modelo.

Exemplos no varejo:
- `fato_vendas` → **uma linha por item de nota** (InvoiceNo + StockCode). Grain atômico.
- `fato_vendas_diaria` → **uma linha por dia x país** (agregado). Grain diário.
- `foto_estoque_diaria` → **uma linha por produto x dia** (snapshot).

**Perguntas para declarar o grain**:
- A linha pode ser duplicada? Se sim, o grain está errado.
- Posso responder 'quantas linhas há por pedido'? Se não, o grain esconde detalhe.
- Preciso de histórico por item ou só total do dia?

**Regra de ouro da Medallion**: a Prata guarda o **grain mais atômico** que o negócio precisa. O Ouro agrega a partir dela. Nunca agregue cedo demais — detalhe perdido não volta.


### 💻 Na prática — Declarando o grain do projeto (varejo)

Vamos explicitar os 3 grains que usaremos:


In [ ]:
# Os 3 grains do nosso projeto — documente assim em todo projeto
grains = {
    "fato_vendas (Prata)":       "1 linha = 1 ITEM de 1 NOTA (atômico) — InvoiceNo+StockCode",
    "fato_vendas_diaria (Ouro)": "1 linha = 1 DIA x 1 PAIS (agregado) — data_venda+Country",
    "foto_estoque (Prata)":      "1 linha = 1 PRODUTO x 1 DIA (snapshot)",
}
for k, v in grains.items():
    print(f"{k:30s} → {v}")
print("\nGrain atômico na Prata = máxima flexibilidade; agregados no Ouro = performance.")

## 📖 Teoria — 6. O teste do grain — 3 perguntas que salvam o projeto

Antes de criar qualquer tabela, responda:

1. **'Se eu contar linhas, o que estou contando?'** — Se a resposta não for cristalina, o grain está vago.
2. **'Posso ter duas linhas com a mesma chave natural?'** — Se sim, falta coluna na PK.
3. **'Se eu agregar para um grain maior, perco informação essencial?'** — Se sim, mantenha o atômico.

No nosso `fato_vendas`, a PK é `(InvoiceNo, StockCode, InvoiceDate)` no Bronze; na Prata vira surrogate keys + medidas. Duplicata nesse grain = erro de ingestão (capturado por expectation).


## 📖 Teoria — 7. Tabelas Fato — o coração que bate

**Fato** = evento de negócio mensurável. Características:
- **Muitas linhas** (centenas de milhões é normal).
- **Poucas colunas textuais** — quase tudo é chave estrangeira + medida numérica.
- **Cresce sempre** — append-only por natureza (como o Bronze, mas limpo).
- **Chave primária** é composta pelas FKs (ou surrogate da fato, se precisar).

Cada linha **deve** ter: FKs para todas as dimensões + pelo menos 1 medida. Fato sem medida = **factless** (ver §8).


## 📖 Teoria — 8. Os 5 tipos de fato — quando usar cada um

| Tipo | O que registra | Exemplo no varejo | Quando usar |
|---|---|---|---|
| **Transaction** | Um evento pontual | `fato_vendas` (cada item vendido) | Quase sempre — é o grain atômico |
| **Periodic Snapshot** | Estado em intervalo fixo | `foto_estoque_diaria` (estoque todo dia) | Monitorar evolução (estoque, saldo) |
| **Accumulating Snapshot** | Ciclo de vida com datas móveis | `fato_pedido` (pedido → pagamento → envio → entrega, 1 linha que é atualizada) | Pipeline com marcos |
| **Factless (coverage)** | Evento sem medida | `fato_promocao_produto` (produto esteve em promoção?) | Cobertura, participação, elegibilidade |
| **Consolidated / Aggregate** | Fato de fatos | `fato_vendas_diaria` (Ouro) | Performance de BI |

**Regra**: comece sempre por **Transaction**. Os outros são derivados ou complementares.


## 📖 Teoria — 9. Additivity — a matemática que decide se a soma faz sentido

Nem toda medida pode ser somada livremente:

- **Aditiva** — soma em qualquer dimensão sem distorcer. Ex.: `quantidade`, `receita`. Pode `SUM` por dia, país, produto.
- **Semi-aditiva** — soma em algumas dimensões, não em outras. Ex.: `saldo_em_estoque` (soma por produto OK, por dia NÃO — tem que pegar o último dia). `AVG` é semi-aditivo.
- **Não-aditiva** — nunca soma direto. Ex.: `percentual`, `taxa_conversao`, `ticket_medio`. Tem que recalcular: `SUM(receita)/COUNT(pedidos)`.

> 🎯 **Pegadinha de prova e de entrevista**: 'Posso somar ticket médio por país para ter a média geral?' — NÃO. Média de médias distorce. Recalcule sempre.


### 💻 Na prática — Classificando as medidas do varejo

Vamos explicitar a aditividade de cada medida:


In [ ]:
# Aditividade das medidas do fato_vendas
medidas = {
    "quantidade":      "ADITIVA — SUM(quantidade) por qualquer dimensão",
    "receita":          "ADITIVA — SUM(quantidade*preço) por qualquer dimensão",
    "saldo_estoque":    "SEMI-ADITIVA — SUM por produto OK, por tempo pegue o último snapshot",
    "ticket_medio":     "NAO-ADITIVA — recalcule SUM(receita)/COUNT(pedidos)",
    "taxa_desconto":    "NAO-ADITIVA — recalcule a cada consulta",
}
for k, v in medidas.items():
    print(f"{k:18s} → {v}")

## 📖 Teoria — 10. Tabelas Dimensão — o contexto que dá significado

**Dimensão** = o 'quem, o que, onde, quando, como, por quê' do fato.
- **Poucas linhas** (milhares, não milhões).
- **Muitas colunas textuais** — atributos descritivos, hierarquias.
- **Chave primária = surrogate key** (`sk_*`).
- **Muda lentamente** — por isso existe SCD.

Boa dimensão é **larga e rasa**: muitas colunas, poucas linhas, desnormalizada de propósito para evitar joins em cadeia no BI.


## 📖 Teoria — 11. Star vs Snowflake vs Galaxy — a geometria do modelo

- **Star (estrela)** ★ — fato no centro, dimensões desnormalizadas ao redor. 1 join por dimensão. **Padrão do Databricks e da prova**. Rápido, simples para o usuário.
- **Snowflake (floco)** ❄ — dimensões normalizadas em sub-dimensões (ex.: `dim_produto` → `dim_categoria` → `dim_departamento`). Economiza espaço, mas cria joins em cadeia. Evite, a menos que a dimensão seja gigante e compartilhada.
- **Galaxy / Constellation** 🌌 — múltiplos fatos compartilhando dimensões conformadas (ex.: `fato_vendas` e `fato_estoque` compartilham `dim_produto` e `dim_tempo`). É o **DW completo**: vários Stars conectados.

```
         dim_cliente
             │
dim_tempo ──fato_vendas── dim_produto          ← STAR (1 fato)
             │
         dim_loja

  fato_vendas ─┐
               ├─ dim_produto (conformed) ─┐  ← GALAXY (2 fatos)
  foto_estoque ─┘                          │
               └─ dim_tempo  (conformed) ──┘
```


### 💻 Na prática — O Star do projeto — e o Galaxy que vamos construir

Hoje: 1 Star (`fato_vendas`). No curso: Galaxy com `foto_estoque` compartilhando dimensões.


In [ ]:
# Star atual do projeto (Prata)
print("STAR: fato_vendas no centro")
print("  ├─ dim_cliente  (sk_cliente, CustomerID, Country, ... )")
print("  ├─ dim_produto  (sk_produto, StockCode, Description, categoria)")
print("  ├─ dim_tempo    (sk_tempo, data, ano, mes, trimestre)")
print("  └─ dim_loja     (sk_loja, Country → futuro: loja física)")
print("\nGALAXY futuro: fato_vendas + foto_estoque compartilham dim_produto e dim_tempo")

## 📖 Teoria — 12. Os 9 tipos de dimensão que você precisa conhecer

| Tipo | O que é | Exemplo | Quando usar |
|---|---|---|---|
| **Conformed** | Dimensão compartilhada entre fatos (mesma chave e atributos) | `dim_tempo` usada por vendas e estoque | Sempre que possível — é o que integra o DW |
| **Role-Playing** | Mesma dimensão com papéis diferentes | `dim_data` como `data_pedido`, `data_envio`, `data_entrega` | Reutilize sem duplicar tabela |
| **Junk (garbage)** | Agrupa flags/indicadores de baixa cardinalidade | `dim_transacao` (tipo_pagamento, canal, indicador_fraude) | Evita Fato com 15 colunas booleanas |
| **Degenerate** | Dimensão sem tabela (atributo direto na fato) | `InvoiceNo` (número da nota) | Chave operacional sem atributos |
| **Outrigger** | Dimensão de dimensão (snowflake pontual) | `dim_cliente` → `dim_endereco` | Atributo gigante e volátil (endereço) |
| **Bridge** | Resolve N:N | `bridge_cliente_programa` (cliente ↔ programas de fidelidade) | Fato precisa contar por múltiplos valores |
| **Mini-dim** | Fatia atributos voláteis | `dim_cliente_profile` (faixa de crédito volátil) | Evita SCD2 explosivo |
| **Heterogeneous (combo)** | Supertabela de subtipos | `dim_produto` (livro tem ISBN, roupa tem tamanho) | Subtipos com atributos distintos |
| **Shrunken / Rollup** | Dimensão agregada | `dim_mes` (só ano/mês) | Fato agregado mensal |

> 💡 No curso, você verá na prática: **conformed** (semana 4), **junk** e **degenerate** (semana 5), **role-playing** (semana 8), **bridge** e **mini-dim** (§15).


## 📖 Teoria — 13. Hierarquias — o mapa de navegação do usuário

Toda dimensão tem hierarquias que o usuário usa para drill-down:
- `dim_tempo`: ano → trimestre → mês → dia
- `dim_produto`: departamento → categoria → subcategoria → produto
- `dim_loja`: país → região → cidade → loja

Tipos:
- **Balanceada**: todo ramo tem mesma profundidade (tempo).
- **Ragged / variável**: ramos com profundidades diferentes (organograma).
- **Skip-level (ragged)**: níveis pulados (produto sem subcategoria).

**Best practice**: mantenha hierarquias **achatadas (flattened)** na dimensão (uma coluna por nível), não normalizadas. O BI agradece. No Delta, use `COMMENT` para documentar cada nível.


## 📖 Teoria — 14. Chaves — natural, surrogate e durable (NK / SK / DK)

- **Natural Key (NK)**: o ID do sistema de origem (`CustomerID=17850`). Instável: pode mudar, ser reutilizado, ter formato diferente por fonte.
- **Surrogate Key (SK)**: inteiro sequencial artificial (`sk_cliente=42`). Estável, compacto, imune a mudanças da origem. **PK da dimensão** e **FK da fato**.
- **Durable Key (DK) / BKCC**: chave estável da entidade ao longo do tempo, mesmo com SCD2 (ex.: `dk_cliente`). Permite contar 'quantos clientes únicos' ignorando versões.

**Regra de ouro**: Fatos referenciam **SK**, nunca NK. A NK fica na dimensão como `nk_cliente` para rastreabilidade.

```
dim_cliente
  sk_cliente  (PK, surrogate, ex.: 42)
  nk_cliente  (NK, ex.: '17850' do ERP)
  dk_cliente  (DK, ex.: 'C17850' estável)
  nome, país, ...

fato_vendas
  sk_cliente  (FK → dim_cliente.sk_cliente)
  sk_produto  (FK → dim_produto.sk_produto)
  quantidade, receita  (medidas)
```


### 💻 Na prática — Criando surrogate keys no Lakehouse — 3 estratégias

Vamos comparar as 3 formas de gerar SK e quando usar cada uma:


In [ ]:
# Estratégia 1: row_number (simples, para carga full)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sha2, col, monotonically_increasing_id
df_dim = spark.table("workspace.bronze.vendas_bronze") \
    .select("CustomerID", "Country").dropDuplicates(["CustomerID"])
df_dim_sk1 = df_dim.withColumn("sk_cliente", row_number().over(Window.orderBy("CustomerID")))
df_dim_sk1.show(3)
print("row_number: simples, mas muda a cada recarga full — não é estável para SCD2.")

In [ ]:
# Estratégia 2: hash da NK (determinística, idempotente)
df_dim_sk2 = df_dim.withColumn("sk_cliente", sha2(col("CustomerID"), 256))
df_dim_sk2.show(3, truncate=False)
print("sha2(NK): determinística — mesmo cliente sempre gera mesma SK. Boa para idempotência.")

In [ ]:
# Estratégia 3: sequence + MERGE (incremental, estável, padrão SCD2 em produção)
# Na Prata SCD2 real (semana 8), a SK é gerada por `APPLY CHANGES INTO` (Delta Live Tables)
# ou por sequence + MERGE. Nunca recalcule SK de dimensões SCD2 com row_number.
print("Em produção SCD2: use APPLY CHANGES / sequence — SK estável entre cargas.")

## 📖 Teoria — 15. O erro mais caro: usar NK na fato

Se a fato referencia NK e o cliente muda de código (fusão de sistemas), todas as linhas antigas ficam órfãs. Com SK, a dimensão mapeia NK→SK e a fato continua íntegra. Além disso, SK inteiro é **10–100x mais rápido** para joins que string.


## 📖 Teoria — 16. SCD — Slowly Changing Dimensions (o capítulo inteiro)

Dimensões mudam devagar (cliente muda de cidade, produto muda de categoria). Como registrar?

| Tipo | Estratégia | Perde histórico? | Complexidade | Quando usar |
|---|---|---|---|---|
| **SCD 0** | Nunca muda (fixo) | N/A | Zero | Atributos imutáveis (data de nascimento) |
| **SCD 1** | Sobrescreve | ✅ Sim | Baixa | Correção de erro — histórico não importa |
| **SCD 2** | Nova linha com vigência | ❌ Não | Alta | Auditoria / análise temporal (padrão) |
| **SCD 3** | Coluna de valor anterior | Parcial | Média | Só precisa do 'antes e agora' (ex.: `cidade_atual`, `cidade_anterior`) |
| **SCD 4** | Tabela de histórico separada | ❌ Não | Alta | Quer manter dimensão 'atual' magra |
| **SCD 6** | SCD1 + SCD2 + SCD3 (híbrido) | ❌ Não | Muito alta | Precisa do atual sobrescrito E do histórico |
| **SCD 7** | SCD1 na dim + SCD2 em outrigger | ❌ Não | Alta | Atributos voláteis vs estáveis separados |

> 🎯 **Dica de prova/interview**: 90% das perguntas são SCD1 vs SCD2. Resposta-padrão: 'Correção → SCD1; histórico/auditoria → SCD2; só anterior→ SCD3.' Na Medallion, SCD2 vive na **Prata**.


## 📖 Teoria — 17. SCD2 em detalhes — as colunas que não podem faltar

Uma dimensão SCD2 tem, além dos atributos, estas colunas técnicas:

| Coluna | Papel | Exemplo |
|---|---|---|
| `sk_*` | PK surrogate da versão | 101, 102 |
| `nk_*` | NK da origem | '17850' |
| `valid_from` | Início da vigência | 2024-01-01 |
| `valid_to` | Fim da vigência (NULL = atual) | NULL |
| `is_current` | Flag de versão atual | true/false |
| `row_hash` | Hash dos atributos (detectar mudança) | sha2(concat(*), 256) |

Consulta 'estado atual': `WHERE is_current = true`. Consulta 'como era em 2024-06-01': `WHERE '2024-06-01' BETWEEN valid_from AND coalesce(valid_to, '9999-12-31')`.


### 💻 Na prática — SCD2 — antes e depois (visual)

Veja o efeito de um SCD2 quando o cliente 17850 muda de país:


In [ ]:
# SCD2: estado ANTES da mudança
print("dim_cliente (SCD2) — ANTES:")
print("sk | nk    | país           | valid_from | valid_to | current")
print(" 1 | 17850 | United Kingdom | 2010-12-01 | NULL     | true")
print("\nSCD2: DEPOIS de mudar para France em 2024-06-15:")
print("sk | nk    | país           | valid_from | valid_to   | current")
print(" 1 | 17850 | United Kingdom | 2010-12-01 | 2024-06-14 | false")
print(" 2 | 17850 | France         | 2024-06-15 | NULL       | true")
print("\nFatos antigos apontam para sk=1 (UK); fatos novos para sk=2 (France).")
print("Relatórios históricos continuam corretos — sem reescrever a fato.")

## 📖 Teoria — 18. Como implementar SCD2 no Databricks — as 3 opções

**Opção A — `APPLY CHANGES INTO` (Delta Live Tables, padrão da Medallion)** — declarativa, gerencia `valid_from/to/is_current` sozinha. É o que usamos na Semana 8:
```python
@dlt.table
def dim_cliente_scd2():
    return dlt.apply_changes(
        target='dim_cliente', source='stg_cliente',
        keys=['nk_cliente'], sequence_by='updated_at',
        stored_as_scd_type=2)
```
**Opção B — `MERGE` + lógica de vigência** — manual, para notebooks sem DLT.
**Opção C — `DeltaTable.merge` (Python)** — mesma lógica, API DataFrame.

Nunca faça SCD2 com `overwrite` — você apagaria o histórico.


### 💻 Na prática — SCD1 vs SCD2 — quando dói escolher errado

Um caso real para fixar:


**Cenário**: o cliente 17850 aparece como `United Kingdom` em 2010 e `France` em 2024.

- **Se usar SCD1** (sobrescreve): todo o histórico de 2010–2024 passa a dizer `France`. Relatório de 'vendas por país em 2011' fica errado. Mas é simples e a dimensão tem 1 linha por cliente.
- **Se usar SCD2** (nova linha): 2010–2024 continua `UK` (sk=1), 2024+ vira `France` (sk=2). Histórico preservado, mas a dimensão dobra de tamanho e todo join precisa considerar `is_current`.

**Decisão**: Relatórios históricos precisam estar corretos? → SCD2. Correção de cadastro? → SCD1. No varejo, `dim_cliente` (endereço) = SCD2; `dim_produto` (descrição corrigida) = SCD1.

## 📖 Teoria — 19. SCD avançados que caem em entrevista sênior

- **SCD4**: mantém `dim_cliente` (só atuais) + `dim_cliente_hist` (todas as versões). A fato aponta para a atual; quem precisa de histórico join com a hist. Mantém a dimensão 'quente' pequena.
- **SCD6 (1+2+3)**: guarda `cidade_atual` (SCD1, sempre atualizado), `cidade_hist` (SCD2, vigência) e `cidade_anterior` (SCD3). Permite responder 'onde o cliente mora hoje' e 'onde morava na época da venda' na mesma linha.
- **SCD7**: separa atributos voláteis (ex.: `status_credito`) em outrigger SCD2, mantendo o núcleo SCD1.

Você não precisa implementar SCD4/6/7 no curso — mas **saber explicar** mostra senioridade.


## 📖 Teoria — 20. Casos avançados que separam júnior de sênior

**1. Late-arriving dimension** — o fato chega antes da dimensão (venda com `CustomerID` que ainda não existe em `dim_cliente`). Solução: criar linha 'inferred member' (só NK, resto NULL) e preencher depois (SCD1).

**2. Late-arriving fact** — fato antigo chega atrasado (venda de 2024-01 chega em março). Solução: inserir na fato e **recalcular agregados do Ouro** (não só append).

**3. Many-to-many (N:N)** — venda com múltiplos vendedores. Solução: **bridge table** + **weight factor**.
```
fato_vendas ── bridge_venda_vendedor ── dim_vendedor
  receita=100       vendedor A (peso 0.6) → 60
                    vendedor B (peso 0.4) → 40
```
Sem weight, a receita seria contada 2x.

**4. Factless + Bridge** — `fato_aluno_curso` (aluno fez quais cursos?) sem medida, com bridge para múltiplos cursos.

**5. Mini-dimensão** — atributos voláteis (ex.: `score_credito` que muda todo mês) viram `dim_cliente_mini` separada, evitando SCD2 explosivo em `dim_cliente`. A fato referencia as duas.

**6. Heterogeneous products** — produtos com atributos distintos (livro tem ISBN, roupa tem tamanho). Solução: **dim_produto** com colunas genéricas + `dim_produto_livro` / `dim_produto_roupa` (subtipo) ou JSON Variant.


### 💻 Na prática — Bridge N:N na prática — venda com 2 vendedores

Veja como o weight evita dupla contagem:


In [ ]:
# Fato (1 venda) + bridge (2 vendedores) + weight
fato = [("NF001", 100.0)]  # 1 venda, receita 100
bridge = [("NF001", "VEND_A", 0.6), ("NF001", "VEND_B", 0.4)]
# Receita por vendedor = receita * weight
for _, vend, w in bridge:
    print(f"{vend}: {100*w:.0f} (100 x {w})")
print("\nSem weight, cada vendedor pareceria ter vendido 100 — total 200 (errado).")
print("Com weight, total continua 100 — correto.")

## 📖 Teoria — 21. Medallion × Modelagem — onde cada coisa vive

| Camada | Conteúdo | Modelagem | Chave | SCD | Exemplo |
|---|---|---|---|---|---|
| **Bronze** | Dado cru, como chegou | Nenhuma — espelho da origem | NK da origem | Não | `workspace.bronze.vendas_bronze` (append-only, `_ingested_at`) |
| **Prata** | Dado limpo, tipado, deduplicado | **Star completo**: dimensões + fatos atômicos | **SK** | **SCD1/2 aqui** | `workspace.prata.dim_cliente` (SCD2), `workspace.prata.fato_vendas` (grain atômico) |
| **Ouro** | Agregados de negócio | **Marts denormalizados** (1 Star → 1 mart) | SK do agregado | Não (derivado) | `workspace.ouro.vendas_por_dia`, `workspace.ouro.receita_por_pais` |

**Fluxo**: Bronze (raw) → Prata (dimensional, SCD) → Ouro (mart, agregado). Essa é a frase que você repete em entrevista: 'Bronze append-only, Prata dimensional SCD, Ouro mart agregado.'


### 💻 Na prática — O que NÃO fazer em cada camada

Regras que salvam o projeto:


- **Bronze NUNCA** modela: sem joins, sem SK, sem SCD, sem `DROP`. É o 'HD externo' — se perder, não tem como reconstituir.
- **Prata NUNCA** agrega: grain atômico + dimensões. Se agregar na Prata, você perde detalhe para o Ouro.
- **Ouro NUNCA** recebe dado cru: só consome Prata. Se o Ouro lê Bronze direto, você fura a governança e a linhagem.
- **Nenhuma camada lê 'para trás'**: Ouro não alimenta Prata; Prata não alimenta Bronze. Fluxo é **sempre para frente**.

## 📖 Teoria — 22. Prata em detalhes — o Data Warehouse dimensional dentro do Lakehouse

A Prata do nosso projeto tem:

```
dim_cliente  (SCD2)  — sk_cliente PK, nk_cliente, nome, país, valid_from/to, is_current
dim_produto  (SCD1)  — sk_produto PK, StockCode NK, Description, categoria
dim_tempo             — sk_tempo PK, data, ano, mês, trimestre, dia_semana (conformed)
dim_loja     (SCD1)  — sk_loja PK, Country NK
fato_vendas           — PK = (sk_cliente, sk_produto, sk_tempo, sk_loja) + medidas
foto_estoque (futuro) — snapshot diário, semi-aditivo
```

Tudo **tipado** (TIMESTAMP, não STRING), **deduplicado** (`dropDuplicates`), **validado** (constraints, expectations) e **idempotente** (`overwrite` ou `MERGE` + `APPLY CHANGES`). É aqui que vivem `CHECK (Quantity > 0)` e `NOT NULL (CustomerID)`.


### 💻 Na prática — Prata na prática — o esqueleto que vamos materializar na Semana 4

Este é o código que a Semana 4 executa. Entenda cada linha:


In [ ]:
# ESQUELETO da Prata — Semana 4 materializa isso de verdade
from pyspark.sql.functions import col, row_number, sha2, current_timestamp
from pyspark.sql.window import Window
w = Window.orderBy("CustomerID")
# dim_cliente SCD2 (simplificada — Semana 8 usa APPLY CHANGES)
bronze = spark.table("workspace.bronze.vendas_bronze")
dim_cliente = (bronze.select("CustomerID", "Country").dropDuplicates(["CustomerID"])
    .withColumn("sk_cliente", row_number().over(w))
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to", col("valid_from").cast("timestamp").alias("valid_to"))  # NULL = atual
    .withColumn("is_current", col("valid_from").isNotNull()))
print("dim_cliente: SK + vigência SCD2 (esqueleto)")
print("Na Semana 8, valid_from/to/is_current são geridos por APPLY CHANGES.")

## 📖 Teoria — 23. Ouro em detalhes — os marts que o usuário vê

Se a Prata é o 'estoque organizado', o Ouro é a 'vitrine' — poucas tabelas, nomes de negócio, sem SK exposto (ou com SK simplificado), prontas para `SELECT *`.

| Ouro | Grain | Para quem | Pergunta que responde |
|---|---|---|---|
| `vendas_por_dia` | dia | BI / ML (forecast) | 'Como foi hoje vs ontem?' |
| `receita_por_pais` | país | Diretoria | 'Onde vendemos mais?' |
| `top_produtos` | produto | Compras | 'O que mais vende?' |
| `vendas_por_dia_pais` | dia × país | BI drill-down | 'Como foi o UK ontem?' |

O Ouro é **derivado**: `CREATE OR REPLACE TABLE ... AS SELECT ... FROM Prata GROUP BY ...`. Se a Prata mudar, o Ouro recalcula. Nunca edite Ouro manualmente — ele é **efêmero** por design.


## 📖 Teoria — 24. Implementação no Databricks — Delta, DLT, UC e performance a serviço da modelagem

| Recurso do Databricks | Papel na modelagem |
|---|---|
| **Delta Lake** (`_delta_log`, Time Travel, `MERGE`, CDF) | SCD sem medo: versiona, audita e recupera |
| **DLT `APPLY CHANGES INTO`** | SCD2 declarativo (sem MERGE manual) |
| **Constraints (`CHECK`, `NOT NULL`)** | Garante grain e medidas na Prata (falha a transação se violar) |
| **Expectations (`expect_or_drop`)** | Qualidade no pipeline (ex.: `Quantity > 0`) |
| **Liquid Clustering (`CLUSTER BY`)** | Performance da fato sem particionar demais |
| **Unity Catalog (tags, linhagem, RLS)** | Governança: `PII` tag em `dim_cliente`, lineage Bronze→Ouro |
| **Volumes** | Arquivos de stage (não tabelas) |
| **Genie / Dashboards** | Consumo do Ouro em linguagem natural |

Modelagem sem Delta/UC é **teoria**; com eles é **produção**.


### 💻 Na prática — Delta a favor da modelagem — 3 exemplos que salvam o dia

Rode e entenda por que Delta é o melhor amigo do modelador:


In [ ]:
# 1) Time Travel: "como estava a dimensão ontem?" — auditoria de SCD
df_ontem = spark.read.format("delta").option("timestampAsOf", "2024-06-14") \
    .table("workspace.prata.dim_cliente")
print("Time Travel: dimensão como era em 2024-06-14 (sem recriar nada).")

In [ ]:
# 2) CDF: "o que mudou na dimensão?" — alimenta o Ouro incremental
df_mudancas = spark.read.format("delta").option("readChangeFeed", "true") \
    .option("startingVersion", 5).table("workspace.prata.dim_cliente")
print("CDF: mudanças desde a versão 5 — base para CDC e auditoria.")

In [ ]:
# 3) Constraint: grain violado = transação falha (não dado sujo)
sql_constraint = """
ALTER TABLE workspace.prata.fato_vendas
  ADD CONSTRAINT pk_fato_nulo CHECK (sk_cliente IS NOT NULL AND sk_produto IS NOT NULL);
-- Se uma carga tentar inserir fato sem dimensão, a transação falha inteira (ACID).
"""
print(sql_constraint)

## 📖 Teoria — 25. A Bus Matrix — o mapa do Data Warehouse inteiro em 1 página

A **Dimensional Bus Matrix** cruza **processos de negócio** (linhas) com **dimensões conformadas** (colunas). É o documento que impede o caos: toda nova fato deve reutilizar dimensões já conformadas.

```
                    │ dim_cliente │ dim_produto │ dim_tempo │ dim_loja │ dim_vendedor │
────────────────────┼─────────────┼─────────────┼───────────┼──────────┼──────────────┤
fato_vendas         │      X      │      X      │     X     │    X     │              │
foto_estoque        │             │      X      │     X     │    X     │              │
fato_atendimento    │      X      │             │     X     │    X     │      X       │
```

Se um novo projeto propõe criar `dim_cliente2`, a Bus Matrix grita: 'Use a conformed `dim_cliente`!'


### 💻 Na prática — A Bus Matrix do nosso curso — e seu próximo passo

Vamos desenhar a Bus Matrix do varejo. Este é o exercício que você leva para entrevista:


In [ ]:
# Bus Matrix do projeto (Prata atual + futuro)
import pandas as pd
bus = pd.DataFrame({
    "processo": ["fato_vendas", "foto_estoque_diaria", "fato_devolucao (futuro)"],
    "dim_cliente": ["X", "", "X"],
    "dim_produto": ["X", "X", "X"],
    "dim_tempo":   ["X", "X", "X"],
    "dim_loja":    ["X", "X", ""],
})
display(bus)
print("Toda nova fato deve reutilizar estas dimensões — é a integração do DW.")

## 📖 Teoria — 26. O processo de modelagem — do requisito ao deploy

1. **Entrevistar o negócio** — 'O que você decide com esses dados?' (não 'que relatório você quer?').
2. **Listar processos e grains** — 1 processo = 1 fato.
3. **Desenhar a Bus Matrix** — dimensões conformadas.
4. **Detalhar cada dimensão** — atributos, hierarquias, SCD, fonte.
5. **Detalhar cada fato** — medidas, additivity, FKs.
6. **Prototipar em SQL** — `CREATE TABLE` + `INSERT` de exemplo, validar com o usuário.
7. **Implementar na Prata** — DLT + expectations + `APPLY CHANGES`.
8. **Publicar Ouro e documentar** — `COMMENT ON TABLE`, tags UC, linhagem, dashboard.

Pular direto para o passo 7 é o erro #1 de iniciantes.


## 📖 Teoria — 27. As 20 melhores práticas — checklist de produção

1. Grain atômico na Prata, agregados só no Ouro.
2. Fatos com SK, nunca NK.
3. Dimensões com `sk_*` (PK), `nk_*` e `dk_*` quando houver SCD2.
4. `is_current` + `valid_from/to` em toda SCD2.
5. `row_hash` para detectar mudanças sem comparar 20 colunas.
6. Constraints na Prata: `NOT NULL` em SKs, `CHECK` em medidas.
7. Expectations no DLT: `Quantity > 0`, `CustomerID IS NOT NULL`.
8. Idempotência: `MERGE` ou `APPLY CHANGES`, nunca `append` cego em dimensão.
9. Dimensões conformadas — 1 `dim_tempo`, não 3 cópias.
10. Nomes de negócio no Ouro (`receita_por_pais`, não `agg_02`).
11. `COMMENT ON TABLE/COLUMN` em tudo — documentação viva.
12. Tags UC (`PII`, `financeiro`) + linhagem ativa.
13. `CLUSTER BY` na fato por colunas de filtro (não particione SCD2 por `is_current`).
14. Bridge com `weight` — nunca dupla contagem.
15. Junk dimension para flags — não polua a fato.
16. Teste de grain: `COUNT(*) = COUNT(DISTINCT PK)` deve ser 0 duplicatas.
17. Time Travel habilitado — auditoria gratuita.
18. CDF ligado em dimensões SCD2 (`delta.enableChangeDataFeed = true`).
19. Ouro recalculável: `CREATE OR REPLACE` a partir da Prata, sem estado próprio.
20. Bus Matrix versionada no Git — o contrato do DW.


## 📖 Teoria — 28. Os 7 antipatterns que destroem um Lakehouse

1. **Bronze modelado** — modelar no Bronze impede reprocessamento.
2. **Fato sem grain declarado** — ninguém sabe o que uma linha significa.
3. **NK na fato** — quebra quando a origem muda.
4. **SCD2 sem `is_current`** — todo `SELECT` precisa de `MAX(valid_from)`.
5. **Snowflake desnecessário** — normalizar `dim_produto` em 4 tabelas só para 'economizar' 10 MB.
6. **Ouro lendo Bronze** — fura a Prata, perde SCD e qualidade.
7. **Dimensão gigante sem mini-dim** — `dim_cliente` com 50 colunas voláteis explode em SCD2.

Se você evitar esses 7, já está no top 10% dos modeladores.


### 💻 Na prática — Validação do modelo — as 7 perguntas que todo modelo deve responder

Antes de chamar o modelo de 'pronto', ele deve passar neste teste:


In [ ]:
# Checklist de validação — responda SIM para todas
checklist = [
    "1. Posso explicar o grain em 1 frase sem gaguejar?",
    "2. COUNT(*) = COUNT(DISTINCT PK) — zero duplicatas?",
    "3. Toda FK da fato existe na dimensão? (sem órfãs)",
    "4. Medidas não-aditivas estão documentadas como tal?",
    "5. SCD2 tem is_current + valid_from/to + row_hash?",
    "6. Dimensões são conformed (Bus Matrix)?",
    "7. Um usuário de negócio entende os nomes do Ouro?",
]
for q in checklist:
    print(f"☐ {q}")
print("\nSe algum ☐ ficar em branco, volte ao design.")

## 📖 Teoria — 29. Do modelo ao consumo — BI, Genie, ML e RAG

- **BI (Dashboards)**: Ouro denormalizado → `SELECT * FROM workspace.ouro.vendas_por_dia` sem joins.
- **Genie (linguagem natural)**: Genie lê o Ouro + `COMMENT` das colunas para responder 'qual a receita por país?'.
- **ML (Feature Store)**: features vêm de Prata/Ouro tipadas e SCD-correctas.
- **RAG (Vector Search)**: descrições de `dim_produto` viram chunks com `StockCode` como metadata.

Modelagem bem feita **serve todos** — mal feita, serve ninguém.


## 📖 Teoria — 30. Medallion além do 'Bronze → Prata → Ouro' — as variações que um arquiteto domina

O Medallion de “3 camadas” é o **esqueleto**. Em produção, ele ganha musculatura:

```
  ┌─────────┐
  │ LANDING │  arquivos crus como chegaram (S3/ADLS), sem ACID, com _ingested_at
  └────┬────┘  (às vezes fora do Lakehouse; às vezes Bronze = Landing)
       │ Auto Loader / COPY INTO / Kafka
  ┌────▼────┐
  │  BRONZE │  Delta ACID, schema-on-read, append-only, histórico intocável
  └────┬────┘  vacuuming longo, CDF ligado, tag `pii=true`
       │ DLT / Structured Streaming + Expectations
  ┌────▼────┐
  │  PRATA  │  Star conformed, SCD1/2, tipado, deduplicado, constraints
  └────┬────┘  Liquid Clustering por (sk_tempo, sk_produto)
       │ Gold pipelines (batch ou streaming)
  ┌────▼────┐
  │   OURO  │  Marts por domínio OU tabelões wide — ver estratégias 31–34
  └────┬────┘
       │ BI / ML / RAG / API
  ┌────▼────┐
  │ CONSUMO │  Dashboard, Genie, Feature Store, endpoint
  └─────────┘
```

**Variações comuns**:
- **Bronze = Landing**: simplifica, mas perde o “antes e depois” da ingestão.
- **Quarantine / Dead-letter** entre Bronze→Prata: linhas que falham expectation vão para `quarantine.*` em vez de travar o pipeline.
- **Ouro em 2 níveis**: `ouro_core` (marts canônicos) + `ouro_mart_*` (marts por equipe).


## 📖 Teoria — 31. Estratégia A — Classic Kimball sobre Medallion (a mais usada, a que o curso adota)

**Ideia**: Star conformed na Prata; marts desnormalizados no Ouro; fatos sempre aditivos.

```
  OLTP/SaaS ──► BRONZE (raw) ──► PRATA (Star: dim + fato, SCD2) ──► OURO (marts)
                      │                    │                          │
                      │ _ingested_at       │ sk_*, is_current         │ receita_por_pais
                      │ sem PK             │ constraints              │ vendas_por_dia
```

| Quando usar | Vantagens | Custos |
|---|---|---|
| 80% dos casos; BI + ML precisam do mesmo núcleo | Reuso máximo, Bus Matrix, SCD correto | Ouro precisa ser bem desenhado para não virar “Prata 2” |

**Regra**: 1 Star = 1 processo de negócio (vendas). Novo processo = novo Star que **reusa** dimensões conformadas.


## 📖 Teoria — 32. Estratégia B — Data Vault 2.0 sobre Medallion (auditoria extrema)

- **Bronze**: Vault Raw — Hub (chave de negócio), Link (relação), Satellite (atributos + hash, com `load_date`).
- **Prata**: Vault Business — mesma estrutura + regras de negócio.
- **Ouro**: Stars derivados do Vault (PIT/Bridge para SCD2).

```
  hub_cliente ─┬─ sat_cliente_endereco (hash, load_date)
               └─ link_cliente_pedido ── hub_pedido
```

| Quando usar | Vantagens | Custos |
|---|---|---|
| Auditoria total, múltiplas fontes com chaves conflitantes, linhagem exigida | Rastreabilidade perfeita, paraleliza ingestão | Complexidade alta; precisa gerar Stars no Ouro para BI consumir |



## 📖 Teoria — 33. Estratégia C — One Big Table (OBT) / Wide Table no Ouro

O Ouro vira **1 tabela larga denormalizada** (tudo joinado): `w_vendas` com 80 colunas — produto, cliente, tempo, loja já resolvidos.

```
  Prata (Star) ──► Ouro: w_vendas (fato + dims achatadas)
                   SELECT f.*, c.nome, c.pais, p.categoria, t.ano, t.mes FROM fato_vendas f
                   JOIN dim_cliente c USING (sk_cliente) ...
```

| Quando usar | Vantagens | Custos |
|---|---|---|
| BI self-service que não quer join; export para Excel/Sheets | Uma tabela responde tudo; Genie/RAG adoram | Redundância; precisa recalc toda se Prata muda; não serve para SCD2 histórico fino |

**Híbrido recomendado**: mantenha **marts** (receita_por_pais) **e** 1 OBT para exploração. São complementares.


## 📖 Teoria — 34. Estratégias D e E — Feature Store Gold e Streaming Medallion

**D. Feature Store Gold** — o Ouro vira **feature tables** para ML:
```
  Prata (fato_vendas) ──► Ouro: feature_store.cliente_360 (features por cliente)
                        features: recencia, frequencia, ticket_medio, categoria_favorita
```
Ouro versionado, com `feature_timestamp` e `event_timestamp` (point-in-time correctness). No Databricks: **Feature Store** nativo lê direto do Ouro.

**E. Streaming Medallion** — Bronze e Prata em **Structured Streaming**:
```
  Kafka/Auto Loader (stream) ──► Bronze (streaming table) ──► Prata (streaming + APPLY CHANGES) ──► Ouro (materialized view)
```
Latência de segundos. Use **DLT com `APPLY CHANGES`** e `skipChangeCommits=true` na Prata para não travar com SCD2 em stream.

| Quando usar D | Quando usar E |
|---|---|
| ML em produção precisa de features frescas | Dashboard operacional / detecção de fraude em tempo real |


## 📖 Teoria — 35. Fluxo 1 — Ingestão até o Bronze (batch, streaming e CDC)

**Batch (Auto Loader — padrão do curso)**:
```
  S3: s3://landing/vendas/2024/06/*.csv
       │
       ├─ Auto Loader (cloudFiles) ──► workspace.bronze.vendas_bronze
       │   - `cloudFiles.inferColumnTypes=true`
       │   - `cloudFiles.schemaLocation` para evolução
       │   - expectation: `Quantity > 0`
       └─ quarantine.vendas_rejected (falhas)
```

**Streaming (Kafka/Kinesis)**:
```
  Kafka topic `vendas` ──► readStream.format('kafka') ──► Bronze streaming table (append-only)
```

**CDC (Change Data Capture)** — origem OLTP com `UPDATE/DELETE`:
```
  Debezium/CDC feed ──► Bronze CDC (op, before/after) ──► Prata com APPLY CHANGES (SCD1/2)
```

**Critério do arquiteto**: volume < 1 GB/dia → batch horário; > 1 GB/hora ou SLA < 5 min → streaming; origem com updates → CDC.


### 💻 Na prática — Ingestão na prática — Auto Loader para o Bronze

O padrão que o curso usa (Semana 5). Releia com olhos de arquiteto:


In [ ]:
# Padrão Auto Loader → Bronze (idempotente, com quarantine)
bronze = (spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/_schema/vendas")
  .option("cloudFiles.inferColumnTypes", "true")
  .load("s3://landing/vendas/"))
# Em DLT, o quarantine vira: @dlt.expect_or_fail / expect_or_drop
print("Bronze: Auto Loader + schemaLocation + quarantine = ingestão resiliente")

## 📖 Teoria — 36. Fluxo 2 — Bronze → Prata (onde a modelagem acontece)

```
  workspace.bronze.vendas_bronze (raw, 540k linhas, com _ingested_at)
       │
       ├─ 1. Limpeza: trim, upper, cast, filter Quantity>0  (expectations)
       ├─ 2. Deduplicação: dropDuplicates([InvoiceNo, StockCode])
       ├─ 3. Tipagem: to_timestamp(InvoiceDate), DOUBLE price
       ├─ 4. SK: row_number / sha2 / sequence
       ├─ 5. SCD: dim_cliente (SCD2, APPLY CHANGES), dim_produto (SCD1)
       └─ 6. Fato: join com dims por SK + medidas
            │
            ▼
       workspace.prata.dim_cliente (SCD2)  ← is_current, valid_from/to
       workspace.prata.dim_produto (SCD1)  ← overwrite por StockCode
       workspace.prata.fato_vendas          ← grain atômico, FK = SK
```

**Decisões do arquiteto aqui**:
- Grain atômico ou agregado na Prata? → **Atômico** (flexibilidade).
- Qual dimensão é SCD2? → Só as que precisam de histórico (cliente). O resto SCD1.
- Constraint ou expectation? → Constraint na Prata (falha transação); expectation no DLT (quarantine).


### 💻 Na prática — Prata — o join que materializa o Star

Este é o coração da modelagem. Na Semana 4 ele roda de verdade:


In [ ]:
# Bronze → Prata: join por NK, grava com SK (Star)
fato = (bronze.alias("b")
  .join(dim_cliente.filter("is_current"), bronze.CustomerID == dim_cliente.nk_cliente)
  .join(dim_produto, bronze.StockCode == dim_produto.StockCode)
  .join(dim_tempo, bronze.InvoiceDate == dim_tempo.data)
  .selectExpr("sk_cliente", "sk_produto", "sk_tempo", "Quantity", "UnitPrice", "Quantity*UnitPrice as receita"))
print("Fato com SK: grain preservado, FK íntegra, medida aditiva")

## 📖 Teoria — 37. Fluxo 3 — Prata → Ouro (marts, tabelões e agregados)

```
  workspace.prata.fato_vendas (atômico, 540k linhas)
       │
       ├──► workspace.ouro.vendas_por_dia       (GROUP BY sk_tempo → dia)
       ├──► workspace.ouro.receita_por_pais     (GROUP BY Country)
       ├──► workspace.ouro.w_vendas             (OBT: fato + dims achatadas)
       └──► workspace.ouro.feature_cliente_360  (Feature Store: recencia, frequencia)
```

**Estratégia por domínio (Data Mesh leve)**:
- `ouro_financeiro.*` — time de Finanças é dono.
- `ouro_marketing.*` — time de Marketing é dono.
- Ambos leem da **mesma Prata conformed** — sem duplicar lógica.

**Padrão de refresh**:
- Ouro agregado → `CREATE OR REPLACE TABLE ... AS SELECT ... FROM Prata GROUP BY ...` (recalc total, barato).
- Ouro incremental → `MERGE` por `sk_tempo` (só dia novo).
- Ouro streaming → `MATERIALIZED VIEW` sobre Prata streaming.


### 💻 Na prática — Ouro — 1 Star, 3 consumos diferentes

Mesma Prata, 3 Ouros para 3 públicos:


In [ ]:
# Mesmo fato, 3 Ouros
# 1) Mart agregado (BI): 1 linha por dia
ouro_dia = fato.groupBy("sk_tempo").agg({"receita": "sum"}).withColumnRenamed("sum(receita)", "receita_dia")
# 2) OBT (exploração): 1 linha = 1 venda com tudo achatado
obt = fato.join(dim_cliente, "sk_cliente").join(dim_produto, "sk_produto")  # wide
# 3) Feature (ML): 1 linha por cliente
feat = fato.groupBy("sk_cliente").agg({"receita": "avg", "Quantity": "sum"})
print("3 Ouros, 1 Prata — reuso sem duplicar regra")

## 📖 Teoria — 38. Fluxo 4 — Ouro → Consumo (BI, Genie, ML, RAG, API)

```
  workspace.ouro.* ──┬──► Dashboard (SQL Warehouse, 2X-Small)
                     ├──► Genie (NL → SQL sobre Ouro + COMMENT)
                     ├──► Feature Store → Modelo → Endpoint (Mosaic AI)
                     ├──► Vector Search (dim_produto.Description → chunks)
                     └──► API / DLT downstream (Delta Sharing)
```

O Ouro **nunca** é editado à mão. Ele é **derivado e recalculável**. Se apagar, `REFRESH` recria. Por isso o catálogo marca `COMMENT ON TABLE workspace.ouro.* IS 'MART — derivado de prata.fato_vendas'` e a linhagem UC mostra Bronze → Prata → Ouro automaticamente.


## 📖 Teoria — 39. Matriz de decisão — qual arquitetura escolher?

| Pergunta | Se SIM → | Se NÃO → |
|---|---|---|
| Precisa auditoria total de cada ingestão? | Data Vault no Bronze/Prata | Kimball direto |
| BI quer 1 tabela sem joins? | OBT no Ouro (além dos marts) | Só marts |
| ML precisa de features frescas? | Feature Store Gold (com `event_timestamp`) | Features na Prata bastam |
| Latência < 1 min? | Streaming Medallion (DLT streaming) | Batch (Auto Loader horário) |
| Dimensão tem 50+ atributos voláteis? | Mini-dim + outrigger | Tudo na dim principal |
| Fato tem N:N (venda→vendedores)? | Bridge + weight | Fato simples |

**Resposta de arquiteto em entrevista**: 'Depende do caso de uso — para o varejo do curso, Kimball + OBT + batch é o melhor custo/benefício; se fosse fraude em tempo real, eu usaria Streaming Medallion com Feature Store.'


## 📖 Teoria — 40. Performance e qualidade em cada camada — o checklist do arquiteto

| Camada | Performance | Qualidade | Governança |
|---|---|---|---|
| **Bronze** | `OPTIMIZE` semanal (append-only gera many small files) | `expect_or_drop` (Quantity>0), `quarantine` | Tag `pii=true`, RLS não (ainda cru) |
| **Prata** | `CLUSTER BY (sk_tempo, sk_produto)` na fato; `ZORDER` em dims SCD2 | `CHECK (sk_cliente IS NOT NULL)`, `NOT NULL`, `APPLY CHANGES` | `is_current` indexado, CDF ligado |
| **Ouro** | `OPTIMIZE` + `VACUUM` (Ouro é recriado, retenção curta) | `CHECK (receita >= 0)` nos marts | `GRANT SELECT ON ouro.* TO bi_team` |

**Regra de ouro**: Bronze otimiza **escrita** (append rápido); Prata otimiza **join** (SK); Ouro otimiza **leitura** (agregado, poucas linhas).


### 💻 Na prática — Cluster e constraints — onde cada um mora

Decore este mapa para a prova e para o design review:


In [ ]:
# Onde cada otimização vive (arquiteto decide na criação da tabela)
print("Bronze: TBLPROPERTIES (delta.enableChangeDataFeed=true) — para SCD2 futuro")
print("Prata: CLUSTER BY (sk_tempo) na fato; CONSTRAINT CHECK (sk_cliente IS NOT NULL)")
print("Ouro:   CLUSTER BY (Country) em receita_por_pais; VACUUM retain 0 HOURS (recalculável)")
print("\nPrata = join rápido; Ouro = scan rápido; Bronze = ingestão rápida")

## 📖 Teoria — 41. Governança e observabilidade — o que o arquiteto não esquece

- **Unity Catalog**: 1 catálogo por ambiente (`workspace` na Free, `prod`/`dev` em conta paga); `GRANT` no Ouro, não no Bronze.
- **Linhagem**: `LINEAGE` mostra `vendas_bronze → fato_vendas → w_vendas` automaticamente (Delta CDF).
- **Expectations (DLT)**: `expect_or_fail` no Bronze (trava pipeline se dado cru vier quebrado), `expect_or_drop` na Prata.
- **Observabilidade**: `DESCRIBE HISTORY` + `AUDIT LOG` + métricas de DLT (linhas in/out/quarantine).

Arquiteto que não desenha governança junto com modelagem entrega um castelo sem portas.


### 💻 Na prática — O pipeline completo em 1 diagrama — cole no README

Este é o diagrama que um arquiteto apresenta no design review:


```mermaid
flowchart LR
  A[Fontes: OLTP, SaaS, CSV] --> B[LANDING S3]
  B -->|Auto Loader| C[BRONZE Delta]
  C -->|DLT + Expectations| D[PRATA Star]
  D --> E1[OURO Mart Dia]
  D --> E2[OURO OBT w_vendas]
  D --> E3[OURO Feature Store]
  E1 & E2 & E3 --> F[Consumo: BI / Genie / ML / RAG]
  C -.->|quarantine| G[Dead Letter]
```

> 💡 No Databricks: **DLT** orquestra `LANDING → BRONZE → PRATA → OURO` com `APPLY CHANGES` para SCD2 e `expectations` para quarantine — tudo declarativo.

> 🎯 **Dica de prova**: A DEA cobra: grain, fato vs dimensão, Star vs Snowflake, SCD1 vs SCD2. A DEP cobra: APPLY CHANGES (SCD2), bridge com weight, late-arriving, e Medallion × modelagem. Decore: 'Prata = Star com SCD, Ouro = mart agregado; SCD1 corrige, SCD2 historia; bridge precisa weight; late dimension = inferred member.'


## 🎯 Exercícios de fixação

**1.** Declare o grain de `fato_vendas` e de `foto_estoque_diaria` em 1 frase cada.

**2.** Classifique: `dim_cliente`, `fato_vendas`, `dim_produto`, `foto_estoque` — quais são fatos e por quê?

**3.** Crie uma junk dimension `dim_transacao` com 3 flags do varejo (ex.: é_presente, canal, tipo_pagamento). Quantas linhas ela teria?

**4.** Desenhe o Star do varejo com `fato_vendas` no centro e 4 dimensões. Depois estenda para Galaxy adicionando `foto_estoque`.

**5.** Cliente 17850 muda de 'United Kingdom' para 'France' em 2024-06-15. Escreva as 2 linhas SCD2 (sk, nk, país, valid_from, valid_to, is_current).

**6.** Venda NF001 tem 2 vendedores (60%/40%) e receita 100. Sem weight, qual o total por vendedor? Com weight?

**7.** Onde na Medallion vivem: (a) SCD2, (b) agregados por país, (c) dado cru com `_ingested_at`? Justifique.

**8.** Preencha a Bus Matrix: linhas = fato_vendas, foto_estoque, fato_atendimento; colunas = dim_cliente, dim_produto, dim_tempo, dim_loja, dim_vendedor.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Grain

`fato_vendas`: 1 linha = 1 item de 1 nota (atômico). `foto_estoque_diaria`: 1 linha = 1 produto × 1 dia (snapshot).

**2.** Fato vs Dimensão

Fatos: `fato_vendas` e `foto_estoque` (eventos/estados mensuráveis, muitas linhas, medidas). Dimensões: `dim_cliente` e `dim_produto` (contexto, poucas linhas, atributos). `foto_estoque` é fato mesmo com 1 linha por produto/dia — mede estoque.

**3.** Junk

Ex.: `dim_transacao(is_presente, canal, tipo_pagamento)` — 2×3×3=18 combinações possíveis, mas só as que ocorrem viram linhas (ex.: 8). A fato referencia `sk_transacao` em vez de 3 colunas booleanas.

**4.** Star → Galaxy

Star: fato_vendas no centro, dim_cliente/produto/tempo/loja ao redor. Galaxy: adicione foto_estoque compartilhando dim_produto e dim_tempo (conformed). Desenhe os dois fatos ligados às mesmas dimensões.

**5.** SCD2

Antes: (1, 17850, UK, 2010-12-01, 2024-06-14, false). Depois: (2, 17850, France, 2024-06-15, NULL, true). Fatos antigos → sk=1; novos → sk=2.

**6.** Bridge weight

Sem weight: cada vendedor = 100 (total 200, errado). Com weight: A=60, B=40 (total 100, correto). Weight evita dupla contagem.

**7.** Medallion

(a) Prata — Star com SCD (dimensões versionadas). (b) Ouro — mart agregado denormalizado. (c) Bronze — append-only, sem modelagem. Fluxo sempre Bronze→Prata→Ouro.

**8.** Bus Matrix

fato_vendas: X em cliente/produto/tempo/loja. foto_estoque: X em produto/tempo/loja. fato_atendimento: X em cliente/tempo/loja/vendedor. Produto e tempo são conformed (X em 2+ linhas).



## ✅ Checklist de fechamento

- [ ] Expliquei OLTP vs OLAP e Kimball vs Inmon em 2 minutos cada.
- [ ] Declarei o grain do projeto e validei com as 3 perguntas.
- [ ] Domino os 5 tipos de fato, 9 tipos de dimensão e 20 boas práticas.
- [ ] Implemento SCD1/2/3 e entendo SCD4/6/7 para entrevistas sênior.
- [ ] Mapeei Bronze (raw) → Prata (Star SCD) → Ouro (mart) na Medallion.
- [ ] Preenchi a Bus Matrix e sei evitar os 7 antipatterns.
- [ ] Pronto para materializar a Prata na Semana 4 e o SCD2 na Semana 8.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*